In [2]:
!nvidia-smi

Sun Apr 12 08:50:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
!pip install ultralytics opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 38.5 MB/s eta 0:00:00


In [6]:
from ultralytics import YOLO

# Load the model
model = YOLO('yolov8n.pt')

# Print the dictionary of all 80 class IDs and their corresponding names
print(model.names)

{0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'airplane', 5: 'bus', 6: 'train', 7: 'truck', 8: 'boat', 9: 'traffic light', 10: 'fire hydrant', 11: 'stop sign', 12: 'parking meter', 13: 'bench', 14: 'bird', 15: 'cat', 16: 'dog', 17: 'horse', 18: 'sheep', 19: 'cow', 20: 'elephant', 21: 'bear', 22: 'zebra', 23: 'giraffe', 24: 'backpack', 25: 'umbrella', 26: 'handbag', 27: 'tie', 28: 'suitcase', 29: 'frisbee', 30: 'skis', 31: 'snowboard', 32: 'sports ball', 33: 'kite', 34: 'baseball bat', 35: 'baseball glove', 36: 'skateboard', 37: 'surfboard', 38: 'tennis racket', 39: 'bottle', 40: 'wine glass', 41: 'cup', 42: 'fork', 43: 'knife', 44: 'spoon', 45: 'bowl', 46: 'banana', 47: 'apple', 48: 'sandwich', 49: 'orange', 50: 'broccoli', 51: 'carrot', 52: 'hot dog', 53: 'pizza', 54: 'donut', 55: 'cake', 56: 'chair', 57: 'couch', 58: 'potted plant', 59: 'bed', 60: 'dining table', 61: 'toilet', 62: 'tv', 63: 'laptop', 64: 'mouse', 65: 'remote', 66: 'keyboard', 67: 'cell phone', 68: 'microw

In [34]:
import cv2
import numpy as np
from ultralytics import YOLO
from concurrent.futures import ThreadPoolExecutor

# Load the YOLOv8 Nano model
model = YOLO('yolov8n.pt')

CLOSE_THRESHOLD_HEIGHT_RATIO = 0.45

def run_yolo_pipeline(frame):
    """Deep Learning Thread: Handles dynamic object detection."""
    # Create a copy of the frame for thread safety
    img = frame.copy()
    frame_height = img.shape[0]
    results = model(img, conf=0.4, verbose=False)

    detections = [] # Store drawing instructions to apply later

    for result in results:
        boxes = result.boxes
        for box in boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            cls_id = int(box.cls[0])
            conf = float(box.conf[0])
            class_name = model.names[cls_id]

            if class_name in ['person', 'car', 'motorcycle', 'bus', 'truck']:
                obj_width = x2 - x1
                obj_height = y2 - y1

                # Aspect ratio filter for "pole persons"
                if class_name == 'person' and (obj_width / obj_height) < 0.20:
                    continue

                height_ratio = obj_height / frame_height

                if height_ratio > CLOSE_THRESHOLD_HEIGHT_RATIO:
                    distance_estimate = "CLOSE (Stop)"
                    color = (0, 0, 255)
                else:
                    distance_estimate = "FAR (Safe)"
                    color = (0, 255, 0)

                label = f"{class_name} {conf:.2f} - {distance_estimate}"
                detections.append({'coords': (x1, y1, x2, y2), 'color': color, 'label': label})

    return detections


def run_classical_pipeline(frame):
    """Classical CV Thread: Handles vector-optimized barrier detection with Contour Filtering."""
    img = frame.copy()
    height, width = img.shape[:2]

    # ==========================================
    # 1. THE BLINDERS (ROI)
    # ==========================================
    roi_x1 = int(width * 0.25)
    roi_x2 = int(width * 0.70)
    roi_y1 = int(height * 0.35)
    roi_y2 = int(height * 0.70)

    roi_mask_img = np.zeros((height, width), dtype=np.uint8)
    cv2.rectangle(roi_mask_img, (roi_x1, roi_y1), (roi_x2, roi_y2), 255, -1)

    # ==========================================
    # 2. THE COLOR MASK
    # ==========================================
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    # Keeping the forgiving bounds so we definitely see the barrier
    lower_red1, upper_red1 = np.array([0, 120, 100]), np.array([10, 255, 255])
    lower_red2, upper_red2 = np.array([160, 120, 100]), np.array([180, 255, 255])

    mask1 = cv2.inRange(hsv, lower_red1, upper_red1)
    mask2 = cv2.inRange(hsv, lower_red2, upper_red2)
    red_mask = cv2.bitwise_or(mask1, mask2)
    red_mask = cv2.bitwise_and(red_mask, red_mask, mask=roi_mask_img)


    # ==========================================
    # 3. DILATION (The Smear)
    # ==========================================
    # Now that ONLY the massive barrier blocks are left, stretch them together
    horizontal_kernel = np.ones((1, 100), np.uint8)
    red_mask = cv2.dilate(red_mask, horizontal_kernel, iterations=1)

    # ==========================================
    # 4. CONTOUR FILTERING (The Geometry Sniper)
    # ==========================================
    contours, _ = cv2.findContours(red_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    clean_mask = np.zeros_like(red_mask)

    for cnt in contours:
        # 1. Get the area
        area = cv2.contourArea(cnt)

        # 2. Get the bounding box dimensions (Width and Height)
        x, y, w, h = cv2.boundingRect(cnt)

        # Prevent division by zero errors
        if h == 0: continue

        # 3. Calculate Aspect Ratio
        aspect_ratio = float(w) / h

        # THE GAUNTLET:
        # - area > 500: Kills tiny scattered streaks
        # - w > 200: Demands the object is physically very wide
        # - aspect_ratio > 3.0: Demands the object is a horizontal line, killing chunky noise
        if area > 500 and w > 200 and aspect_ratio > 3.0:
            cv2.drawContours(clean_mask, [cnt], -1, 255, -1)

    red_mask = clean_mask

    # ==========================================
    # 5. EDGE & LINE DETECTION
    # ==========================================
    edges = cv2.Canny(red_mask, 50, 150)
    lines = cv2.HoughLinesP(edges, 1, np.pi/180, threshold=50, minLineLength=150, maxLineGap=100)

    barrier_detected = False
    barrier_line = None

    if lines is not None:
        lines = lines[:, 0, :]
        x1, y1 = lines[:, 0], lines[:, 1]
        x2, y2 = lines[:, 2], lines[:, 3]

        angles = np.abs(np.arctan2(y2 - y1, x2 - x1) * 180.0 / np.pi)
        valid_lines = lines[(angles < 25) | (angles > 155)]

        if len(valid_lines) > 0:
            barrier_detected = True
            barrier_line = valid_lines[0]

    return barrier_detected, barrier_line, red_mask


# ==========================================
# MAIN EXECUTION
# ==========================================

video_path = '/content/PXL_20250325_044746327.TS.mp4'
cap = cv2.VideoCapture(video_path)
fps = int(cap.get(cv2.CAP_PROP_FPS))
out_width = 1280
out_height = 720
output_path = 'output_threaded_hybrid.mp4'
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_path, fourcc, fps, (out_width, out_height))

frame_count = 0
needs_darkening = False

# Initialize the Thread Pool with 2 workers (one for YOLO, one for CV)
with ThreadPoolExecutor(max_workers=2) as executor:

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            print("Processing complete. Video saved to:", output_path)
            break
        frame = cv2.resize(frame, (1280, 720))
        frame_count += 1
        # Only check the brightness once every 15 frames
        if frame_count % 15 == 0:
            tiny_frame = cv2.resize(frame, (64, 64))
            gray = cv2.cvtColor(tiny_frame, cv2.COLOR_BGR2GRAY)
            mean_brightness = np.mean(gray)

            # Update our toggle state
            needs_darkening = (mean_brightness > 180)

        # Apply the fix based on the current toggle state
        if needs_darkening:
            frame = cv2.convertScaleAbs(frame, alpha=1.3, beta=-80)


        # Submit both pipelines to run concurrently on the current frame
        future_yolo = executor.submit(run_yolo_pipeline, frame)
        future_cv = executor.submit(run_classical_pipeline, frame)

        # Wait for both threads to finish and grab their results
        yolo_detections = future_yolo.result()
        # Grab the mask from the thread results
        barrier_detected, barrier_line, debug_mask = future_cv.result()

        # # Convert the black & white mask to BGR so the VideoWriter doesn't crash
        # debug_frame = cv2.cvtColor(debug_mask, cv2.COLOR_GRAY2BGR)

        # # Write the DEBUG mask to the video instead of the normal frame
        # out.write(debug_frame)

        # Draw YOLO results
        for det in yolo_detections:
            x1, y1, x2, y2 = det['coords']
            color = det['color']
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(frame, det['label'], (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

        #Draw CV results
        if barrier_detected:
            vx1, vy1, vx2, vy2 = barrier_line
            cv2.line(frame, (vx1, vy1), (vx2, vy2), (255, 0, 0), 4)
            cv2.putText(frame, "STOP - BARRIER DETECTED", (50, 100), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 0, 255), 4)

        #Write the unified frame to the output video
        out.write(frame)

cap.release()
out.release()
cv2.destroyAllWindows()

Processing complete. Video saved to: output_threaded_hybrid.mp4
